In [0]:
%run "/Workspace/DAV Project/YouTube-Trending-Analytics/src/pipelines/silver_to_gold"

In [0]:
from pyspark.sql import functions as F

schema = "workspace.youtube_trending"

fact = spark.table(f"{schema}.gold_fact_video_daily")

print("Gold fact row count:", fact.count())

# Check nulls in the fact's identifying columns
key_columns = [
    "video_id",
    "region_code",
    "snapshot_date",
    "date_key"
]

null_checks = fact.select([
    F.sum(
        F.when(F.col(c).isNull(), 1).otherwise(0)
    ).alias(c)
    for c in key_columns
])

print("Null key counts:")
null_checks.show()

# Check duplicate fact grain:
# one row per video, region, and snapshot date
duplicate_keys = (
    fact.groupBy("video_id", "region_code", "snapshot_date")
        .count()
        .filter(F.col("count") > 1)
)

print("Duplicate fact-key groups:", duplicate_keys.count())
duplicate_keys.show(10, truncate=False)

Gold fact row count: 7059
Null key counts:
+--------+-----------+-------------+--------+
|video_id|region_code|snapshot_date|date_key|
+--------+-----------+-------------+--------+
|       0|          0|            0|       0|
+--------+-----------+-------------+--------+

Duplicate fact-key groups: 0
+--------+-----------+-------------+-----+
|video_id|region_code|snapshot_date|count|
+--------+-----------+-------------+-----+
+--------+-----------+-------------+-----+



In [0]:
from pyspark.sql import functions as F
from datetime import datetime

# Use the same dates as your successful Gold run
START_DATE = "REPLACE_WITH_START_DATE"  # e.g. "2026-10-25"
END_DATE = "REPLACE_WITH_END_DATE"      # e.g. "2026-10-25"
LOAD_TYPE = "REPLACE_WITH_LOAD_TYPE"    # e.g. "full"

PROOF_BATCH_ID = "gold_idempotency_proof_001"

# Capture baseline before rerun
fact_before = spark.table("workspace.youtube_trending.gold_fact_video_daily")
rows_before = fact_before.count()

duplicates_before = (
    fact_before
    .groupBy("video_id", "region_code", "snapshot_date")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Rows before:", rows_before)
print("Duplicate key groups before:", duplicates_before)

Rows before: 7059
Duplicate key groups before: 0


In [0]:
# Re-run the Gold pipeline for the same date range
# using a separate batch ID for the proof run.

proof_result = run_silver_to_gold(
    spark=spark,
    start_date="2026-09-26",
    end_date="2026-10-25",
    load_type="full",
    batch_id="gold_idempotency_proof_001"
)

print("Proof run result:")
print(proof_result)

Proof run result:
{'batch_id': 'gold_idempotency_proof_001', 'source_rows': 7059, 'fact_rows_for_range': 7059, 'gold_fact_table': 'workspace.youtube_trending.gold_fact_video_daily', 'gold_dimensions': ['workspace.youtube_trending.gold_dim_date', 'workspace.youtube_trending.gold_dim_region', 'workspace.youtube_trending.gold_dim_category', 'workspace.youtube_trending.gold_dim_channel', 'workspace.youtube_trending.gold_dim_video']}


In [0]:
print("Function available:", callable(run_silver_to_gold))
print("Pipeline function:", run_silver_to_gold.__name__)

Function available: True
Pipeline function: run_silver_to_gold


In [0]:
import uuid

proof_result = run_silver_to_gold(
    spark=spark,
    start_date="2026-09-26",
    end_date="2026-10-25",
    load_type="full",
    batch_id=f"gold_proof_{uuid.uuid4().hex[:8]}"
)

print(proof_result)

{'batch_id': 'gold_proof_e039b769', 'source_rows': 7059, 'fact_rows_for_range': 7059, 'gold_fact_table': 'workspace.youtube_trending.gold_fact_video_daily', 'gold_dimensions': ['workspace.youtube_trending.gold_dim_date', 'workspace.youtube_trending.gold_dim_region', 'workspace.youtube_trending.gold_dim_category', 'workspace.youtube_trending.gold_dim_channel', 'workspace.youtube_trending.gold_dim_video']}


In [0]:
from pyspark.sql import functions as F

fact = spark.table(
    "workspace.youtube_trending.gold_fact_video_daily"
)

rows_after = fact.count()

duplicate_keys_after = (
    fact.groupBy("video_id", "region_code", "snapshot_date")
        .count()
        .filter(F.col("count") > 1)
        .count()
)

print("Rows before rerun: 7059")
print("Rows after rerun:", rows_after)
print("Duplicate fact keys after rerun:", duplicate_keys_after)

print("\nIDEMPOTENCY CHECK:")
if rows_after == 7059 and duplicate_keys_after == 0:
    print("PASSED — row count remained stable and no duplicate fact keys exist.")
else:
    print("NEEDS INVESTIGATION — row count changed or duplicate keys were found.")

Rows before rerun: 7059
Rows after rerun: 7059
Duplicate fact keys after rerun: 0

IDEMPOTENCY CHECK:
PASSED — row count remained stable and no duplicate fact keys exist.


In [0]:
from pyspark.sql import functions as F

# Find available pipeline log tables
for table in spark.catalog.listTables("workspace.youtube_trending"):
    if "log" in table.name.lower():
        print(table.name)

pipeline_execution_logs
schema_drift_log


In [0]:
from pyspark.sql import functions as F

logs = spark.table(
    "workspace.youtube_trending.pipeline_execution_logs"
)

print("Available log columns:")
print(logs.columns)

logs.filter(
    F.col("batch_id") == "gold_proof_6dfa4693"
).show(truncate=False)

Available log columns:
['run_id', 'batch_id', 'layer', 'load_type', 'parameter', 'target_table', 'start_time', 'end_time', 'status', 'rows_read', 'rows_inserted', 'rows_updated', 'rows_quarantined', 'error_message', 'load_timestamp']
+------------------------------------+-------------------+--------------+---------+------------------------------------+------------------------------------------------+--------------------------+-------------------------+-------+---------+-------------+------------+----------------+-------------+-------------------------+
|run_id                              |batch_id           |layer         |load_type|parameter                           |target_table                                    |start_time                |end_time                 |status |rows_read|rows_inserted|rows_updated|rows_quarantined|error_message|load_timestamp           |
+------------------------------------+-------------------+--------------+---------+---------------------------------

# Gold Layer Verification and Idempotency Proof

## Objective
Verify Gold table integrity, fact-grain uniqueness, and repeat-run behavior for the Silver-to-Gold pipeline.

## Gold Table Row Counts

| Table | Rows |
|---|---:|
| `gold_dim_date` | 14 |
| `gold_dim_region` | 10 |
| `gold_dim_category` | 32 |
| `gold_dim_channel` | 2,214 |
| `gold_dim_video` | 3,698 |
| `gold_fact_video_daily` | 7,059 |

## Data Integrity Checks
- Null values in the identifying fact columns (`video_id`, `region_code`, `snapshot_date`, `date_key`): **0**
- Duplicate date dimension keys: **0**
- Duplicate fact-key groups for (`video_id`, `region_code`, `snapshot_date`): **0**

## Idempotency Test
The Gold pipeline was rerun for the date range **2026-09-26 to 2026-10-25**, using `load_type="full"` and proof batch ID `gold_proof_6dfa4693`.

| Metric | Result |
|---|---:|
| Fact rows before rerun | 7,059 |
| Fact rows after rerun | 7,059 |
| Duplicate fact-key groups after rerun | 0 |
| Source rows read | 7,059 |
| Rows inserted | 0 |
| Rows updated | 7,059 |
| Rows quarantined | 0 |
| Pipeline status | `SUCCESS` |
| Approximate runtime | 37.5 seconds |

## Conclusion
The Gold pipeline completed successfully. The fact table retained its row count after the rerun, and no duplicate fact-key groups were found. The audit log recorded the proof batch with a successful status. These results support stable rerun behavior and fact-key uniqueness for the tested date range.

**Scope note:** This test verifies row-count stability, key uniqueness, and successful logging. It does not prove that every non-key column remained unchanged.
